In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Simple Playwright Example

In [ ]:
#import elecc_colombia.playwright_example as pe
import asyncio
from playwright.async_api import async_playwright

async with async_playwright() as pw:

    # --- 1. Launch the browser -------------------------------------------
    # headless=False  → a real browser window opens so you can watch
    # slow_mo=800     → adds 800ms pause between actions (easier to follow)
    # Change headless=True + remove slow_mo once everything works
    browser = await pw.chromium.launch(headless=False, slow_mo=800)
    page = await browser.new_page()
    
            # --- 2. Go to the website --------------------------------------------
    print("Navigating to site...")
    await page.goto("https://books.toscrape.com/catalogue/page-1.html")
    #await page.goto("https://wapp.registraduria.gov.co/electoral/2026/presidente-de-la-republica/")

    # wait_for_load_state ensures the page is fully rendered before we act
    await page.wait_for_load_state("networkidle")

    print(f"  Title : {await page.title()}")
    print(f"  URL   : {page.url}")
    
    # --- 4. Click the "Next" button --------------------------------------
    print("\nLooking for the Next button...")
    
    # CSS selector: <li class="next"><a href="...">next</a></li>
    next_btn = await page.query_selector("li.next a")
    
    if not next_btn:
        print("  Button not found — check your selector!")
        await browser.close()
        exit(1)

    print("  Found it! Clicking...")
    await next_btn.click()
    
    # Wait for the new page to fully load after the click
    await page.wait_for_load_state("networkidle")
    
    # --- 5. Confirm we're now on page 2 ----------------------------------
    print(f"\nAfter clicking Next:")
    print(f"  Title : {await page.title()}")
    print(f"  URL   : {page.url}")   # should show page-2.html
    
    book_elements_p2 = await page.query_selector_all("article.product_pod h3 a")
    print(f"\nFirst 3 books on page 2:")
    
    for el in book_elements_p2[:3]:
        title = await el.get_attribute("title")
        print(f"  • {title}")   
    
    # --- 6. Close the browser -------------------------------------------
    #input("\nPress Enter to close the browser...")   # pause so you can inspect
    await browser.close()
    print("Done!")

Navigating to site...
  Title : All products | Books to Scrape - Sandbox
  URL   : https://books.toscrape.com/catalogue/page-1.html

Looking for the Next button...
  Found it! Clicking...

After clicking Next:
  Title : All products | Books to Scrape - Sandbox
  URL   : https://books.toscrape.com/catalogue/page-2.html

First 3 books on page 2:
  • In Her Wake
  • How Music Works
  • Foolproof Preserving: A Guide to Small Batch Jams, Jellies, Pickles, Condiments, and More: A Foolproof Guide to Making Small Batch Jams, Jellies, Pickles, Condiments, and More
Done!
